# 01 — Exploratory Data Analysis
FRAUDSCOPE AI. Goal: understand the class imbalance and check that the behavioural signals
(amount, hour, device, location, velocity) actually separate fraud from legitimate activity.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from src.preprocessing import build_events
from src.features import build_features

os.chdir("..")   # run from the project root so data/ paths resolve

## 1. Load the canonical event table

In [ ]:
events = build_events(use_synthetic=True)   # set False to use data/raw/train_transaction.csv
events.head()

In [ ]:
print(events.shape)
print(events.is_fraud.value_counts())
print(f"fraud rate: {events.is_fraud.mean():.3%}")

## 2. Class imbalance
This is the single most important fact about the dataset: it decides how we train and how we
evaluate. Accuracy is useless here.

In [ ]:
ax = events.is_fraud.value_counts().plot(kind="bar", color=["#12A150", "#D13438"])
ax.set_xticklabels(["legitimate", "fraud"], rotation=0)
ax.set_title("Class distribution"); plt.show()

## 3. Amount distribution by class

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(data=events.assign(log_amt=np.log1p(events.amount)),
            x="log_amt", hue="is_fraud", common_norm=False, fill=True, ax=ax)
ax.set_xlabel("log(1 + amount)"); ax.set_title("Amount by class"); plt.show()

events.groupby("is_fraud").amount.describe()[["mean", "50%", "max"]]

## 4. Hour-of-day pattern

In [ ]:
hours = (events.timestamp // 3600) % 24
tab = pd.crosstab(hours, events.is_fraud, normalize="columns")
tab.plot(figsize=(9, 4)); plt.xlabel("hour of day")
plt.title("When do transactions happen?"); plt.show()

## 5. Behavioural features: do they separate the classes?

In [ ]:
X, y, profiles = build_features(events, save=False)
summary = X.assign(is_fraud=y.values).groupby("is_fraud").mean().T
summary["ratio"] = summary[1] / summary[0].replace(0, np.nan)
summary.sort_values("ratio", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["amount_ratio", "txn_count_10min", "is_night"]):
    X.assign(is_fraud=y.values).groupby("is_fraud")[col].mean().plot(
        kind="bar", ax=ax, color=["#12A150", "#D13438"])
    ax.set_title(col); ax.set_xticklabels(["legit", "fraud"], rotation=0)
plt.tight_layout(); plt.show()

## 6. Correlation between engineered features

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Feature correlation"); plt.show()

## Takeaways
1. Fraud is a small minority class → evaluate with PR-AUC / precision / recall, never accuracy.
2. Relative amount (`amount_ratio`) separates the classes better than the raw amount, which
   justifies per-customer behavioural profiling.
3. Night hours, new devices, new locations and velocity all shift between classes, but none of
   them is decisive alone — which is exactly why we combine them in a model plus a risk engine.